# Crowd Digital Twin — Kaggle CrowdHuman Fine-Tuning Pipeline
Trains YOLO11s and YOLO11n on CrowdHuman dataset for dense crowd person & head detection.

In [ ]:
# Cell 1: Environment Setup
!pip install -q ultralytics supervision opencv-python-headless pyyaml tqdm pandas

In [ ]:
# Cell 2: Verify GPU
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())

In [ ]:
# Cell 3: Convert CrowdHuman ODGT Annotations to YOLO format
import os
from pathlib import Path

# Kaggle dataset input path check
input_path = Path('/kaggle/input/crowdhuman-dataset') if Path('/kaggle/input/crowdhuman-dataset').exists() else Path('./CrowdHuman')
print('CrowdHuman Source Path:', input_path)

# Prepare dataset
!python training_scripts/prepare_crowdhuman.py --data-dir {input_path} --out-dir ./dataset_crowdhuman --labels both

In [ ]:
# Cell 4: Fine-Tune YOLO11s Model on CrowdHuman (640x640)
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
results_11s_640 = model.train(
    data='./dataset_crowdhuman/crowdhuman.yaml',
    epochs=35,
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    name='yolo11s_crowdhuman_640',
    exist_ok=True
)
print('YOLO11s 640 Training Complete!')

In [ ]:
# Cell 5: Fine-Tune YOLO11s Model at High Resolution (960x960)
model_960 = YOLO('yolo11s.pt')
results_11s_960 = model_960.train(
    data='./dataset_crowdhuman/crowdhuman.yaml',
    epochs=35,
    imgsz=960,
    batch=8,
    workers=4,
    device=0,
    name='yolo11s_crowdhuman_960',
    exist_ok=True
)
print('YOLO11s 960 Training Complete!')

In [ ]:
# Cell 6: Save Trained Weights Artifacts
!cp runs/detect/yolo11s_crowdhuman_640/weights/best.pt ./yolo11s_crowdhuman.pt
print('Saved trained weights to ./yolo11s_crowdhuman.pt')